<a href="https://colab.research.google.com/github/CarlosMendez1997Col/Automation_of_satellite_image_downloads_GeeMap_APIs_REST/blob/main/2.%20CHIRPS/Automate_Download_Precipitation_CHIRPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# `Automated script to download CHIRPS precipitation images using GeeMap`



# Import libraries and packages

In [1]:
!pip install geemap
!pip install earthengine-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.7 MB/s eta 0:00:00


In [27]:
import ee
import geemap
import requests
import os
import shutil
import geopandas as gpd
from google.colab import files
import webbrowser
import ipywidgets as widgets
from datetime import datetime
import datetime
import zipfile
import io

## Autentication in Google Colab and GEE

In [3]:
auth_url = ee.Authenticate(auth_mode='notebook')
webbrowser.open(auth_url)
ee.Authenticate()
ee.Initialize()

To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=VeZ7wDxzTuqmBEIeiGN-Laq6mccVod0J-YjGp_K7t6M&tc=IWbEALxWf42B4AfOKssgHjJSM-WwVCpY-IGSqXHlcTA&cc=qstbHb_f64QtGo7b5idNO2zX8e_SXWug1caW-PcyaG4

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AfrIepBN3kgnnOiCUSHvHaQjHIxNseD_AV8W0vW-gGEah06XtP6tiN5nsM4

Successfully saved authorization token.


## Create basemaps and draw Area of Interest (AOI)

In [4]:
MapAOI = geemap.Map()
MapAOI.setOptions('HYBRID')
MapAOI.add_draw_control()
MapAOI.setCenter(-74.0817, 4.6097, 5)

styles = {
    "OpenStreetMap": "OpenStreetMap",
    "Esri World Street Map": "Esri.WorldStreetMap",
    "Esri World Imagery": "Esri.WorldImagery",
    "Esri World Topo Map": "Esri.WorldTopoMap",
    "Esri World Gray Canvas": "Esri.WorldGrayCanvas",
    "Esri World Shaded Relief": "Esri.WorldShadedRelief",
    "Esri World Terrain": "Esri.WorldTerrain",
    "CartoDB DarkMatter": "CartoDB.DarkMatter",
    "CartoDB Positron": "CartoDB.Positron",
    "CartoDB Voyager": "CartoDB.Voyager",
    "OpenTopoMap": "OpenTopoMap"
}

style_dropdown = widgets.Dropdown(
    options=list(styles.keys()),
    value="Esri World Imagery",
    description="Basemap:"
)
def update_style(change):
    MapAOI.add_basemap(styles[change['new']])
style_dropdown.observe(update_style, names='value')

display(style_dropdown)
MapAOI

Dropdown(description='Basemap:', index=2, options=('OpenStreetMap', 'Esri World Street Map', 'Esri World Image…

Map(center=[4.6097, -74.0817], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDa…

In [5]:
aoi_import = ee.Geometry.Polygon(
    [[[-75.81533, 9.018015],
      [-75.81533, 9.237671],
      [-75.643688, 9.237671],
      [-75.643688, 9.018015],
      [-75.81533, 9.018015]]],
    geodesic=False
)

aoi = MapAOI.user_roi
if aoi:
    print("AOI definido por el usuario:")
    print(aoi.getInfo())
else:
    print("AOI no definido por el usuario. Usando AOI por defecto:")
    aoi = aoi_import
    print(aoi_import.getInfo())

AOI no definido por el usuario. Usando AOI por defecto:
{'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-75.81533, 9.018015], [-75.81533, 9.237671], [-75.643688, 9.237671], [-75.643688, 9.018015], [-75.81533, 9.018015]]]}


# Search CHIRPS precipitation images in AOI

In [13]:
collection = ee.ImageCollection("UCSB-CHC/CHIRPS/V3/PENTAD") \
    .filterBounds(aoi) \
    .filterDate("2017-05-26", "2026-01-26")

def clip_bbox(image):
    return image.clip(aoi)

collection = collection.map(clip_bbox)

info_list = []
for image in collection.getInfo()["features"]:
    img_id = image.get("id")
    props = image.get("properties", {})

    if "system:time_start" in props:
        ts = props["system:time_start"] / 1000
        fecha = datetime.datetime.utcfromtimestamp(ts).strftime("%Y-%m-%d")
    else:
        fecha = None

    info_list.append({"id": img_id,"date": fecha,"properties": props})

print("Number of images in collection:", len(info_list))
for info in info_list:
    print(info["date"], info["id"], props["system:index"])

Number of images in collection: 624
2017-05-26 UCSB-CHC/CHIRPS/V3/PENTAD/20170526 20260121
2017-06-01 UCSB-CHC/CHIRPS/V3/PENTAD/20170601 20260121
2017-06-06 UCSB-CHC/CHIRPS/V3/PENTAD/20170606 20260121
2017-06-11 UCSB-CHC/CHIRPS/V3/PENTAD/20170611 20260121
2017-06-16 UCSB-CHC/CHIRPS/V3/PENTAD/20170616 20260121
2017-06-21 UCSB-CHC/CHIRPS/V3/PENTAD/20170621 20260121
2017-06-26 UCSB-CHC/CHIRPS/V3/PENTAD/20170626 20260121
2017-07-01 UCSB-CHC/CHIRPS/V3/PENTAD/20170701 20260121
2017-07-06 UCSB-CHC/CHIRPS/V3/PENTAD/20170706 20260121
2017-07-11 UCSB-CHC/CHIRPS/V3/PENTAD/20170711 20260121
2017-07-16 UCSB-CHC/CHIRPS/V3/PENTAD/20170716 20260121
2017-07-21 UCSB-CHC/CHIRPS/V3/PENTAD/20170721 20260121
2017-07-26 UCSB-CHC/CHIRPS/V3/PENTAD/20170726 20260121
2017-08-01 UCSB-CHC/CHIRPS/V3/PENTAD/20170801 20260121
2017-08-06 UCSB-CHC/CHIRPS/V3/PENTAD/20170806 20260121
2017-08-11 UCSB-CHC/CHIRPS/V3/PENTAD/20170811 20260121
2017-08-16 UCSB-CHC/CHIRPS/V3/PENTAD/20170816 20260121
2017-08-21 UCSB-CHC/CHIRPS/V3

## Visualize latest 3 images of CHIRPS

In [14]:
MapAOIChirps = geemap.Map()
precipitationVis = {"min": 0,"max": 112,"palette": ['#001137', '#0aab1e', '#e7eb05', '#2c7fb8', '#253494']}

images_list_10 = collection.toList(10)

for i in range(3):
    img = ee.Image(images_list_10.get(i))
    date_str = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    MapAOIChirps.addLayer(img.select('precipitation'), precipitationVis, 'prec' + date_str)

MapAOIChirps.addLayer(aoi, {"color":"darkblue", "opacity":0.1}, "AOI")
MapAOIChirps.centerObject(aoi, 12)

MapAOIChirps

Map(center=[9.12783508215922, -75.72950899999893], controls=(WidgetControl(options=['position', 'transparent_b…

In [15]:
download_chirps = collection

## Export CHIRPS images to Google Colab Workspace

In [28]:
os.makedirs("CHIRPS_Exports", exist_ok=True)

def download_images(collection, region):
    images_list = collection.toList(collection.size())
    n = collection.size().getInfo()

    for i in range(n):
        img = ee.Image(images_list.get(i))
        date_str = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd').getInfo()

        if 'precipitation' in img.bandNames().getInfo():
            url_vv = img.select('precipitation').getDownloadURL({
                'scale': 5000,
                'crs': 'EPSG:4326',
                'region': region
            })

            r = requests.get(url_vv)
            z = zipfile.ZipFile(io.BytesIO(r.content))
            z.extractall("CHIRPS_Exports")

            for name in z.namelist():
                if name.endswith(".tif"):
                    os.rename(
                        os.path.join("CHIRPS_Exports", name),
                        os.path.join("CHIRPS_Exports", f"precipitation_{date_str}.tif")
                    )

download_images(download_chirps, aoi)


## Download CHIRPS images

In [30]:
folder_path = "CHIRPS_Exports"
zip_name = "CHIRPS_Exports_ZIP"
shutil.make_archive(zip_name, 'zip', folder_path)
files.download(f"{zip_name}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>